# AI-Assisted Oil Market Intelligence Analysis

This notebook documents the two-mode analytics workflow behind the Streamlit dashboard. The project is a portfolio analytics MVP, not a production SaaS platform, real-time trading system, investment advice tool, causal inference model, or trained trading model.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_sample_prices, load_events
from src.feature_engineering import add_price_features
from src.event_classifier import classify_events
from src.brief_generator import generate_market_brief
from src.event_study import load_verified_events, load_historical_prices, calculate_event_windows

# Part 1: Sample Demo Scenario

This section uses controlled sample prices and sample events for reliable portfolio demonstration. These rows are not live, verified, or investment-grade market data.

In [ ]:
sample_prices = load_sample_prices()
sample_events = load_events()
sample_features = add_price_features(sample_prices)
classified_sample_events = classify_events(sample_events)
sample_prices.head(), classified_sample_events.head()

## Sample Feature Engineering

In [ ]:
sample_features[['date', 'wti_price', 'brent_price', 'wti_ma_7', 'brent_ma_7', 'brent_wti_spread']].tail()

## Sample Price Visualizations

In [ ]:
px.line(sample_features, x='date', y=['wti_price', 'brent_price'], title='Sample WTI and Brent Prices', labels={'value': 'USD per barrel', 'variable': 'Series'})

In [ ]:
px.line(sample_features, x='date', y='brent_wti_spread', title='Sample Brent-WTI Spread', labels={'brent_wti_spread': 'USD per barrel'})

## Sample Event Classification and Counts

In [ ]:
classified_sample_events[['headline', 'category', 'market_signal', 'rule_based_category', 'rule_based_market_signal']].head()

In [ ]:
category_counts = classified_sample_events['category'].value_counts().reset_index()
category_counts.columns = ['category', 'event_count']
px.bar(category_counts, x='category', y='event_count', title='Sample Event Count by Category')

In [ ]:
signal_counts = classified_sample_events['market_signal'].value_counts().reset_index()
signal_counts.columns = ['market_signal', 'event_count']
px.bar(signal_counts, x='market_signal', y='event_count', title='Sample Event Count by Market Signal')

## Template-Based AI-Style Brief

In [ ]:
print(generate_market_brief(sample_features, classified_sample_events.head(8)))

# Part 2: Historical Validation Mode

This section uses actual historical FRED WTI/Brent price data from 2020-01-01 through 2025-12-31 and the user-provided `data/raw/verified_events.csv` file. Results are descriptive and do not prove causality.

In [ ]:
historical_prices = load_historical_prices(refresh=False)
verified_events = load_verified_events()
event_windows = calculate_event_windows(verified_events, historical_prices)
event_windows.to_csv(PROJECT_ROOT / 'data/processed/event_window_results.csv', index=False)
print(f'Historical price rows: {len(historical_prices):,}')
print(f'Verified event rows: {len(event_windows):,}')

## Historical Validation Summary Metrics

In [ ]:
summary_metrics = pd.DataFrame({
    'metric': [
        'verified_events',
        'shifted_event_dates',
        'source_url_coverage',
        'wti_negative_price_windows',
        'extreme_return_flags',
    ],
    'value': [
        len(event_windows),
        int(event_windows['event_date_shifted'].sum()),
        event_windows['source_url_present'].mean(),
        int(event_windows['wti_negative_price_window'].sum()),
        int(event_windows['extreme_return_flag'].sum()),
    ],
})
summary_metrics

## Event Counts

In [ ]:
event_windows['category'].value_counts().reset_index(name='event_count')

In [ ]:
event_windows['expected_market_signal'].value_counts().reset_index(name='event_count')

## Average +5D Returns and Dollar Changes by Category

In [ ]:
category_window_summary = event_windows.groupby('category', as_index=False).agg(
    event_count=('event_id', 'count'),
    avg_wti_return_5d=('wti_return_5d', 'mean'),
    avg_brent_return_5d=('brent_return_5d', 'mean'),
    avg_wti_change_5d=('wti_change_5d', 'mean'),
    avg_brent_change_5d=('brent_change_5d', 'mean'),
).sort_values('event_count', ascending=False)
category_window_summary

In [ ]:
px.bar(category_window_summary, x='category', y='avg_wti_return_5d', title='Average WTI +5D Percentage Return by Category')

In [ ]:
px.bar(category_window_summary, x='category', y='avg_brent_return_5d', title='Average Brent +5D Percentage Return by Category')

In [ ]:
px.bar(category_window_summary, x='category', y='avg_wti_change_5d', title='Average WTI +5D Dollar Change by Category')

In [ ]:
px.bar(category_window_summary, x='category', y='avg_brent_change_5d', title='Average Brent +5D Dollar Change by Category')

## Average WTI +5D Return by Expected Signal

In [ ]:
signal_window_summary = event_windows.groupby('expected_market_signal', as_index=False).agg(
    event_count=('event_id', 'count'),
    avg_wti_return_5d=('wti_return_5d', 'mean'),
    avg_brent_return_5d=('brent_return_5d', 'mean'),
    avg_wti_change_5d=('wti_change_5d', 'mean'),
    avg_brent_change_5d=('brent_change_5d', 'mean'),
)
signal_window_summary

## Source URL Coverage and Anomaly Flags

The current verified event file contains 36 events, and every row has a source URL present. Six event dates were shifted to the next available FRED price date because of weekends, holidays, or missing price observations. Three event windows touch the April 2020 WTI negative-price anomaly, and two windows are flagged as extreme because absolute WTI +5D or +10D returns exceed 100%.

In [ ]:
event_windows.loc[
    event_windows['wti_negative_price_window'],
    ['event_id', 'event_date', 'event_title', 'wti_event_price', 'wti_price_plus_5d', 'wti_return_5d', 'wti_change_5d'],
]

## Portfolio Insights

1. The verified historical sample is intentionally small: 36 events across 2020-2025. The results are useful for portfolio storytelling and analyst workflow design, but they should be read as descriptive event-window summaries, not statistical proof.
2. OPEC / Production and Demand Outlook are the largest categories in the current event file. This makes business sense for crude oil analysis because supply coordination and demand revisions are recurring drivers of market interpretation.
3. Brent returns are less distorted than WTI around the April 2020 negative-price anomaly. The WTI windows around April 2020 create extreme percentage values, while Brent provides a more stable benchmark for cross-checking the same event period.
4. Dollar changes are useful when percentage returns become hard to interpret. For example, WTI moving from a positive price to a negative settlement or rebounding from a negative settlement can produce percentage returns that are mathematically valid but economically awkward.
5. The project demonstrates how verified historical events can be joined with FRED price data to support an analyst-style workflow: load data, clean dates, engineer features, align events to market observations, calculate event windows, and communicate limitations transparently.

## Portfolio Takeaway

This project demonstrates practical analytics skills across data loading, data cleaning, feature engineering, event-window analysis, dashboarding, and AI-assisted market brief generation. The sample mode supports reliable portfolio demos, while historical validation mode shows how verified event data can be connected to actual historical market prices for descriptive analysis.

## Limitations

Historical event-window analysis is exploratory and descriptive. Small sample size, overlapping market drivers, timing uncertainty, market expectations, source completeness, and concurrent macro or geopolitical developments limit causal interpretation. Results should not be treated as trading recommendations or investment advice.